# Custom Iterators

This notebook is about extending the iterator layer of the DSL, not just using it.
A custom iterator clause is the piece that decides **which site tuples are visited**
before predicates and emissions run. If you can control iteration, you can encode
many model families without touching compiler internals.

The goal here is to build an intuition for the contract, then implement two realistic
iterator clauses from scratch and use them in compiled operators.


## Before we start: mental model and contract

When you call an iterator method (for example `for_each_site`), the builder opens a term.
Every later predicate and emission in that term is evaluated over the rows emitted by that iterator.

For custom iterator clauses, the core contract is:

1. subclass `nkdsl.AbstractIteratorClause`
2. implement `build_iterator(self, hilbert, *args, **kwargs)`
3. return data coercible to a `KBodyIteratorSpec`

In practice, the easiest return form is:

- `labels`: a tuple of string labels, such as `("i",)` or `("i", "j")`
- `rows`: a tuple of integer tuples, one tuple per iterator row

The DSL enforces arity consistency (`len(row) == len(labels)`) and expects at least one row.


In [1]:
import jax.numpy as jnp
import netket as nk
import nkdsl

hi = nk.hilbert.Fock(n_max=3, N=6)
print("Hilbert size:", hi.size)
print("Built-in iterator clauses (sample):", sorted(nkdsl.available_iterator_clause_names())[:8])


∣NK⟩ Tip: log(ψ) ∈ ℜ → ψ = exp(logψ) ∈ ℜ₊ (real NN gives only positive wave-function).

Hilbert size: 6
Built-in iterator clauses (sample): ['for_each', 'for_each_distinct_pair', 'for_each_pair', 'for_each_plaquette', 'for_each_site', 'for_each_triplet', 'globally']


## Example 1: iterate only over even sites

This is a simple but very common pattern: select a deterministic subset of sites.
In larger projects, this can represent a sublattice, a gauge partition, or a domain decomposition.

Notice two design choices in the implementation below:

- The clause validates that the generated row list is not empty.
- It takes an optional label argument so you can compose it naturally with existing DSL style.


In [2]:
class ForEachEvenSite(nkdsl.AbstractIteratorClause):
    clause_name = "for_each_even_site"

    def build_iterator(self, hilbert, label: str = "i"):
        n = int(hilbert.size)
        rows = tuple((k,) for k in range(n) if k % 2 == 0)
        if not rows:
            raise ValueError("No even sites are available for this Hilbert space.")
        return (str(label),), rows


nkdsl.register_iterator_clause(ForEachEvenSite, replace=True)
print("Registered:", "for_each_even_site" in nkdsl.available_iterator_clause_names())


Registered: True


### Why this return value works

`return ("i",), ((0,), (2,), (4,), ...)` is accepted because iterator coercion supports
`(labels, over)` as a first-class format.

At compile time, those rows become the static iteration domain for the term.
That means your custom iterator directly controls both term cardinality and the static
padding bound (`max_conn_size`) for the generated operator.


In [3]:
op_even = (
    nkdsl.SymbolicDiscreteJaxOperator(hi, "diag-even")
    .for_each_even_site("i")
    .emit(nkdsl.identity(), matrix_element=nkdsl.site("i").value)
    .build()
    .compile()
)

x = jnp.asarray([0, 2, 1, 3, 2, 0], dtype=jnp.int32)
xp, mels = op_even.get_conn_padded(x)
print("xp shape:", xp.shape)
print("mels:", mels)


xp shape: (3, 6)
mels: [0. 1. 2.]


The nonzero matrix elements correspond to iterator rows that actually exist in the clause.
Since this iterator visits only even indices, odd-site contributions never appear in this term.

That is an important distinction:

- a predicate filters rows **after** iteration
- a custom iterator changes the domain **before** predicates even run


## Example 2: edge-driven iteration with validation

For graph-like models, you often want iteration over a fixed edge list. A custom iterator
is the right layer for that because edge structure is a domain definition, not a boolean filter.

Below we add defensive checks:

- at least one edge must be present
- each endpoint index must be within `[0, hilbert.size)`

This keeps errors close to input and makes broken graphs fail fast.


In [4]:
class ForEachEdge(nkdsl.AbstractIteratorClause):
    clause_name = "for_each_edge"

    def build_iterator(self, hilbert, src: str = "i", dst: str = "j", *, edges):
        n = int(hilbert.size)
        rows = tuple((int(i), int(j)) for i, j in edges)
        if not rows:
            raise ValueError("edges must contain at least one pair.")
        for i, j in rows:
            if i < 0 or j < 0 or i >= n or j >= n:
                raise ValueError(f"edge ({i}, {j}) is out of bounds for hilbert size {n}.")
        return (str(src), str(dst)), rows


nkdsl.register_iterator_clause(ForEachEdge, replace=True)
print("Registered:", "for_each_edge" in nkdsl.available_iterator_clause_names())


Registered: True


In [5]:
edges = [(0, 1), (1, 2), (2, 3), (4, 5)]

op_hop_sym = (
    nkdsl.SymbolicDiscreteJaxOperator(hi, "edge-hop")
    .for_each_edge("i", "j", edges=edges)
    .where((nkdsl.site("i") > 0) & (nkdsl.site("j") < 3))
    .emit(nkdsl.shift("i", -1).shift("j", +1), matrix_element=1.0)
    .build()
)
op_hop = op_hop_sym.compile()

x_hop = jnp.asarray([1, 0, 1, 0, 2, 0], dtype=jnp.int32)
xp_hop, mels_hop = op_hop.get_conn_padded(x_hop)
print("xp_hop shape:", xp_hop.shape)
print("mels_hop:", mels_hop)


xp_hop shape: (4, 6)
mels_hop: [1. 0. 1. 1.]


In [6]:
print(op_hop_sym.to_ir())


symbolic.operator @"edge-hop" [dtype=float64, hermitian=false, hilbert_size=6] {
  ; 1 term(s)

  term #0 "0" [kbody, n_iter=4, max_conn_size=4] {
    iterate: for (i, j) in [(0, 1), (1, 2), (2, 3), ... +1 more]
    where:   ((x[i] > 0) && (x[j] < 3))
    emit #0:
      update:    x'[i] = (x[i] + -1); x'[j] = (x[j] + 1)
      amplitude: 1
  }

}


In the IR, inspect the `iterate:` line first. If the iterator rows are wrong, all downstream
predicate and emission behavior will look wrong too, even if their logic is correct.

For debugging custom iterators, this order usually saves time:

1. verify labels and row arity in the clause
2. print IR and confirm the iterator domain
3. only then debug predicate/emission behavior


## Practical guidelines for production clauses

Keep each iterator clause focused on one selection rule. If a clause needs many optional flags,
it is usually better to split it into two or three explicit clauses with clearer intent.

Prefer deterministic, fully static row generation from builder inputs. The compiler assumes a
static branch shape; dynamic row counts tied to runtime state are not what iterator clauses are for.

Finally, add argument validation in the clause itself. It is much easier for you to fix a clear
`ValueError` from `build_iterator` than to diagnose a silent semantic mismatch several layers later.
